# Simulation Sanity Check

Load a saved simulation pickle, validate its numerical and process-level outputs, and inspect standard figures.

## Execution paths

1. **Inspection (default):** load the selected pickle and run every following section without changing it.
2. **Optional re-run:** set `RERUN_SIMULATION = True`, define any parameter overrides in the dedicated section, and re-run the loaded configuration before validation.

Run the notebook from the campaign directory so pickle discovery starts at the correct location. The selected pickle should contain detailed trajectories; performance-mode results can lack data required by some plots.

**Workflow:**
1. Setup and configuration
2. Load or optionally re-run the simulation
3. KPI, feasibility, and elemental-closure diagnostics
4. Batch, kinetic, and process-level visualizations
5. Interpretation checklist

## 1. Setup and Configuration

In [23]:
from pathlib import Path

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.simulation_data import SimulationData

print("✓ Imports successful")

✓ Imports successful


### 1.1 Discover Available Simulation Pickles

Scan the campaign directory recursively for saved simulation pickles. Configure the selected result and inspection scope in the next cell.

In [24]:
# ── Scan this campaign directory recursively for simulation pickles ───────────
CAMPAIGN_DIR = Path.cwd()
PICKLE_FILES = sorted(
    path for path in CAMPAIGN_DIR.rglob("*.pkl")
    if path.is_file()
)

if not PICKLE_FILES:
    raise FileNotFoundError(
        f"No .pkl files found below the campaign directory: {CAMPAIGN_DIR.resolve()}"
    )

print(f"Found {len(PICKLE_FILES)} simulation pickle(s) below {CAMPAIGN_DIR.resolve()}:\n")
for number, path in enumerate(PICKLE_FILES, start=1):
    print(f"[{number:>3}] {path.relative_to(CAMPAIGN_DIR)}")


Found 25 simulation pickle(s) below C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics:

[  1] 01_FedBatch\export\fedbatch_constant_p2_20260819_0951\opt_20260819_0951.pkl
[  2] 01_FedBatch\export\fedbatch_constant_p2_20260824_2340\opt_20260824_2340.pkl
[  3] 01_FedBatch\export\fedbatch_csfr_p2_20260819_1106\opt_20260819_1106.pkl
[  4] 01_FedBatch\export\fedbatch_csfr_p2_20260825_0001\opt_20260825_0001.pkl
[  5] 01_FedBatch\export\fedbatch_exponential_p2_20260819_1232\opt_20260819_1232.pkl
[  6] 01_FedBatch\export\fedbatch_exponential_p2_20260825_0025\opt_20260825_0025.pkl
[  7] 01_FedBatch\export\fedbatch_gsp_p2_20260819_1707\opt_20260819_1707.pkl
[  8] 01_FedBatch\export\fedbatch_gsp_p2_20260825_0135\opt_20260825_0135.pkl
[  9] 01_FedBatch\export\fedbatch_linear_p2_20260819_1833\opt_20260819_1833.pkl
[ 10] 01_FedBatch\export\fedbatch_linear_p2_20260825_0204\opt_20260825_0204.pkl
[ 11] 02_Batch\export\batch_p2_20260

In [25]:
# ── Select the saved simulation and inspection scope ──────────────────────────
# Set this to one of the numbers printed by the preceding discovery cell.
PICKLE_NUMBER = 12

# Individual batch used by the batch and kinetic figure sections.
BATCH_NUMBER = 1

# None includes every batch in process-level overlays; use an integer to limit them.
MAX_BATCHES = None

# False preserves the loaded result. Set True only to apply overrides and re-run.
RERUN_SIMULATION = False

if not isinstance(PICKLE_NUMBER, int) or isinstance(PICKLE_NUMBER, bool):
    raise TypeError("PICKLE_NUMBER must be an integer.")
if not 1 <= PICKLE_NUMBER <= len(PICKLE_FILES):
    raise ValueError(
        f"PICKLE_NUMBER must be between 1 and {len(PICKLE_FILES)}; "
        f"received {PICKLE_NUMBER}."
    )
if not isinstance(BATCH_NUMBER, int) or isinstance(BATCH_NUMBER, bool) or BATCH_NUMBER < 1:
    raise ValueError("BATCH_NUMBER must be a positive integer.")
if MAX_BATCHES is not None and (
    not isinstance(MAX_BATCHES, int) or isinstance(MAX_BATCHES, bool) or MAX_BATCHES < 1
):
    raise ValueError("MAX_BATCHES must be None or a positive integer.")

PICKLE_PATH = PICKLE_FILES[PICKLE_NUMBER - 1]

print(f"Selected pickle [{PICKLE_NUMBER}]: {PICKLE_PATH.relative_to(CAMPAIGN_DIR)}")
print(f"Batch for detailed plots: {BATCH_NUMBER}")
print(f"Optional re-run enabled: {RERUN_SIMULATION}")

Selected pickle [12]: 02_Batch\export\batch_p2_20260825_0315\opt_20260825_0315.pkl
Batch for detailed plots: 1
Optional re-run enabled: False


## 2. Load the Selected Simulation

This is the default inspection path. The optional re-run section below replaces `sim` and `data` only when enabled.

In [26]:
sim = CellCultureSimulation.load_pickle(PICKLE_PATH, base_export_dir=PICKLE_PATH.parent)
data = sim.data

if not 1 <= BATCH_NUMBER <= len(data.batches):
    raise ValueError(
        f"BATCH_NUMBER must be between 1 and {len(data.batches)} for the selected simulation; "
        f"received {BATCH_NUMBER}."
    )

print(f"✓ Loaded: {data.simulation_type}")
print(f"  Source           : {PICKLE_PATH.resolve()}")
print(f"  Batches          : {len(data.batches)}")
print(f"  Simulation time  : {data.simulated_window_totals_stats.get('total_simulation_time_h', 'N/A'):.1f} h")
print(f"  Parameters name  : {data.parameters.metadata.get('name', 'N/A') if data.parameters else 'N/A'}")

CellCultureSimulation loaded from: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics\02_Batch\export\batch_p2_20260825_0315\opt_20260825_0315.pkl
✓ Loaded: Batch (startup Fedbatch)
  Source           : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics\02_Batch\export\batch_p2_20260825_0315\opt_20260825_0315.pkl
  Batches          : 12
  Simulation time  : 960.0 h
  Parameters name  : base_parameters


### 2.1 Optional Parameter Overrides and Re-run

Leave `RERUN_SIMULATION = False` to inspect the loaded result unchanged. To explore a changed configuration, enable it in the configuration cell and add parameter-path/value pairs to `PARAMETER_OVERRIDES` in the next cell. The re-run replaces `sim` and `data` for every subsequent diagnostic and figure.

### 2.2 Run Optional Overrides

In [27]:
# Keep this mapping empty to re-run the saved configuration without changes.
# Uncomment or add entries only when RERUN_SIMULATION is True.
PARAMETER_OVERRIDES = {
    # "fed_batch.C_Gln_feed": 10,
    # "fed_batch.C_AA_feed": 10,
    # "initial_conditions.C_Glc_init": 40,
    # "initial_conditions.C_Gln_init": 20,
    # "fed_batch.CSFR": 5e-12,
    # "perfusion.CSPR": 10.0e-11,
}

if RERUN_SIMULATION:
    for parameter_path, value in PARAMETER_OVERRIDES.items():
        sim.parameters[parameter_path] = value

    sim = CellCultureSimulation(
        parameters=sim.parameters,
        kinetics=sim.kinetics,
        gas_controller=sim.gas_controller,
        operating_mode=sim.operating_mode,
        feed_strategy=sim.feed_strategy,
        perfusion_strategy=sim.perfusion_strategy,
        stoichiometry_mode=sim.stoichiometry_mode,
        performance_mode=sim.performance_mode,
        solver_method="LSODA",
    )
    data = sim.run_simulation()
    print(f"✓ Re-ran simulation with {len(PARAMETER_OVERRIDES)} override(s).")
else:
    print("✓ Inspection mode: using the loaded simulation unchanged.")

✓ Inspection mode: using the loaded simulation unchanged.


## 3. Validation and Diagnostics

Review optimization feasibility, performance, and elemental closure before interpreting figures.

### 3.1 Optimization Feasibility Check

This uses the same `ObjectiveFunction` path and rejection criteria as differential evolution. A pass means the result is feasible for optimization; it does not replace scientific review.

In [28]:
# Evaluate the current simulation with the same ObjectiveFunction implementation
# used by differential evolution. This avoids maintaining a second, drifting copy
# of the optimization rejection logic in the notebook.

from animakinetica.analysis.optimization import ObjectiveFunction
import numpy as np

WALL_TIMEOUT_S = 10.0  # Match the campaign optimization timeout when diagnosing runtime rejections.

# ObjectiveFunction needs objectives and weights even though this cell only
# evaluates feasibility. Use a harmless available numeric KPI with zero weight.
validation_function = ObjectiveFunction(
    baseline_simulation=sim,
    opt_config={},
    objectives=["STY_kg_m3_day"],
    weights=[0.0],
    timeout=WALL_TIMEOUT_S,
    baseline_values=None,
    debug_rejections=True,
)

# An empty optimization configuration maps this empty vector to the current
# baseline parameters, then runs the same temporary-simulation path and every
# ObjectiveFunction rejection check. Valid evaluations return a non-positive
# score; soft constraint penalties and hard rejections return positive scores.
filter_score = float(validation_function(np.array([], dtype=float)))
passes_filter = filter_score <= 0.0

print("ObjectiveFunction filter check:")
print(f"  {'PASS' if passes_filter else 'FAIL'}  valid ObjectiveFunction score <= 0")
print(f"  Score: {filter_score:.6g}")

# Supplementary diagnostics only; all pass/fail decisions remain in
# ObjectiveFunction. The fields below expose inputs to its current filters.
simulated_totals = data.simulated_window_totals_stats
expected_window_h = validation_function._get_expected_simulation_window_target_h()
soft_violations = validation_function._get_soft_constraint_violations(
    data,
    simulated_totals,
)

try:
    closure = data.compute_boundary_closure()
    elem_res = {
        element: abs(values["relative_residual_pct"])
        for element, values in closure["elements"].items()
    }
except (ValueError, KeyError):
    elem_res = None

total_base_in_L = float(simulated_totals.get("total_base_in_L", 0.0) or 0.0)
total_acid_in_L = float(simulated_totals.get("total_acid_in_L", 0.0) or 0.0)
total_medium_in_L = float(simulated_totals.get("total_medium_in_L", 0.0) or 0.0)
acid_base_fraction = (
    (total_base_in_L + total_acid_in_L) / total_medium_in_L
    if total_medium_in_L > 0.0 else None
)

print("\nDiagnostics:")
print(f"  total_simulation_time_h = {float(simulated_totals.get('total_simulation_time_h', 0.0) or 0.0):.3f}")
print(f"  expected_simulation_window_h = {expected_window_h:.3f}")
print(f"  total_active_cultivation_time_h = {float(simulated_totals.get('total_active_cultivation_time_h', 0.0) or 0.0):.3f}")
print(f"  projection_window_h = {simulated_totals.get('projection_window_h')}")
print(f"  projection_target_time_h = {simulated_totals.get('projection_target_time_h')}")
print(f"  number_of_batches = {simulated_totals.get('number_of_batches')}")
print(f"  total_DCM_kg = {simulated_totals.get('total_DCM_kg', data.summary_stats.get('total_DCM_kg'))}")
print(f"  last_batch_termination_reason = {simulated_totals.get('last_batch_termination_reason')}")
print(f"  unique_batch_termination_reasons = {simulated_totals.get('unique_batch_termination_reasons')}")
print(f"  batch_final_volume_min_fraction = {simulated_totals.get('batch_final_volume_min_fraction')}")
print(f"  batch_final_volume_max_fraction = {simulated_totals.get('batch_final_volume_max_fraction')}")
print(f"  last_batch_final_volume_fraction = {simulated_totals.get('last_batch_final_volume_fraction')}")
print(f"  total_base_in_L = {total_base_in_L:.6g}")
print(f"  total_acid_in_L = {total_acid_in_L:.6g}")
print(f"  total_medium_in_L = {total_medium_in_L:.6g}")
print(f"  acid_base_fraction = {acid_base_fraction if acid_base_fraction is not None else 'N/A'}")
print(f"  phase_active_runtime_h = {simulated_totals.get('phase_active_runtime_h')}")
print(f"  batch_phase_active_runtime_h = {simulated_totals.get('batch_phase_active_runtime_h')}")
print(f"  ObjectiveFunction soft_constraint_violations = {soft_violations}")
if elem_res is not None:
    print(
        "  boundary closure |relative_residual_pct| "
        f"(soft limit {ObjectiveFunction._MAX_CLOSURE_PCT:.0f}%) = {elem_res}"
    )

run_simulation execution time: 659.837 ms
ObjectiveFunction filter check:
  PASS  valid ObjectiveFunction score <= 0
  Score: -0

Diagnostics:
  total_simulation_time_h = 960.000
  expected_simulation_window_h = 960.000
  total_active_cultivation_time_h = 0.000
  projection_window_h = 960.0
  projection_target_time_h = 7680.0
  number_of_batches = 12
  total_DCM_kg = None
  last_batch_termination_reason = Maximum time reached
  unique_batch_termination_reasons = ['Aa depletion', 'Ammonia ic100', 'Maximum time reached']
  batch_final_volume_min_fraction = 1.0
  batch_final_volume_max_fraction = 1.0066818708043086
  last_batch_final_volume_fraction = 1.0
  total_base_in_L = 13.355
  total_acid_in_L = 0
  total_medium_in_L = 20658.7
  acid_base_fraction = 0.0006464592053329185
  phase_active_runtime_h = {'FEDBATCH': 26.165066152844712, 'BATCH': 933.8349338471553}
  batch_phase_active_runtime_h = [{'batch_number': 1, 'BATCH': 98.2670238825751, 'FEDBATCH': 26.165066152844712}, {'batch_numbe

### 3.2 Performance Summary

Inspect the primary KPIs and cultivation-level totals.

In [29]:
print("Performance Metrics:")
for key, value in data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key:<45} {value:.4f}")
    else:
        print(f"  {key:<45} {value}")

Performance Metrics:
  total_DCM_harvested_kg                        97.9045
  total_WCM_harvested_kg                        652.6970
  STY_kg_m3_day                                 0.1530
  STY_WCM_kg_m3_day                             1.0198
  medium_conversion_efficiency_m3_kg            1.6881
  medium_conversion_efficiency_m3_kg_WCM        0.2532
  protein_conversion_efficiency_kg_kg           7.1152
  feed_conversion_efficiency_kg_kg              9.6994
  feed_conversion_efficiency_kg_kg_WCM          1.4549


In [30]:
# Cultivation summary table
data.get_cultivation_summary_df()

,Batch No.,Termination Reason,Cultivation Time (h),Batch Start Time (h),Batch End Time (h),Accum. Cultivation Time (d),Final Glucose (mmol/L),Final Glutamine (mmol/L),Final Lactate (mmol/L),Final Ammonia (mmol/L),...,Accum. Acid Addition Use (L),Accum. Total Inlet Use (L),Harvest Volume (L),Accum. Harvest Volume (L),Final Cell Density (10^6 cells/mL),Final Cell Density (g/L),Cells Harvested (10^12 Cells),Cells Harvested (kg),Accum. Cell Harvest (10^12 Cells),Accum. Biomass Harvest (kg)
0,1,Ammonia ic100,124.432090,0.000000,124.432090,5.184670,1.244628,7.713704,30.140898,8.150000,...,0.0,2013.355196,1728.872732,1728.872732,4.218059,0.632709,7.292488,1.093873,7.292488,1.093873
1,2,Ammonia ic100,82.234103,124.432090,206.666193,8.611091,0.133365,8.784756,23.582596,8.150000,...,0.0,3728.864186,1703.708912,3432.581645,4.050071,0.607511,6.900142,1.035021,14.192630,2.128895
2,3,Ammonia ic100,82.302545,206.666193,288.968738,12.040364,0.105806,8.938254,22.372771,8.150000,...,0.0,5432.573098,1701.496192,5134.077836,4.020049,0.603007,6.840098,1.026015,21.032729,3.154909
3,4,Ammonia ic100,82.304898,288.968738,371.273636,15.469735,0.106293,8.961311,22.184999,8.150000,...,0.0,7134.069290,1701.092094,6835.169931,4.014614,0.602192,6.829229,1.024384,27.861958,4.179294
4,5,Aa depletion,76.416653,371.273636,447.690290,18.653762,0.695477,9.507639,21.949057,7.140419,...,0.0,8835.161385,1692.667989,8527.837919,3.904572,0.585686,6.609144,0.991372,34.471102,5.170665
5,6,Aa depletion,74.166521,447.690290,521.856811,21.744034,0.967816,9.734674,21.768987,6.702354,...,0.0,10527.829373,1690.958571,10218.796490,3.882975,0.582446,6.565949,0.984892,41.037051,6.155558
6,7,Aa depletion,73.362303,521.856811,595.219114,24.800796,1.075219,9.819457,21.684580,6.535246,...,0.0,12218.787944,1690.694472,11909.490962,3.879659,0.581949,6.559318,0.983898,47.596369,7.139455
7,8,Aa depletion,73.071073,595.219114,668.290187,27.845424,1.115967,9.850717,21.650330,6.473104,...,0.0,13909.482416,1690.653629,13600.144592,3.879147,0.581872,6.558294,0.983744,54.154663,8.123199
8,9,Aa depletion,72.964967,668.290187,741.255154,30.885631,1.131099,9.862188,21.637285,6.450222,...,0.0,15600.136046,1690.647312,15290.791904,3.879068,0.581860,6.558135,0.98372,60.712798,9.10692
9,10,Aa depletion,72.926212,741.255154,814.181365,33.924224,1.136667,9.866389,21.632440,6.441830,...,0.0,17290.783358,1690.646335,16981.438239,3.879055,0.581858,6.558111,0.983717,67.270909,10.090636


In [31]:
process_stoich_closure = data.compute_stoichiometric_closure()
process_stoich_closure_df = data.build_closure_summary_df(process_stoich_closure)

process_stoich_biomass = next(
    spec for spec in process_stoich_closure['component_specs']
    if spec['label'] == 'Dry biomass'
)

print('Integrated stoichiometric CHON closure:')
display(process_stoich_closure_df)
print(f"Max absolute elemental residual: {process_stoich_closure['max_abs_residual']:.6g} mol-atoms")

Integrated stoichiometric CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,2751.392759,2751.372590,-0.020168,-0.000733
1,H,5524.111406,5524.064233,-0.047172,-0.000854
2,O,3842.721690,3842.708347,-0.013343,-0.000347
3,N,266.406816,266.399987,-0.006829,-0.002563


Max absolute elemental residual: 0.0471723 mol-atoms


In [32]:
process_external_elemental_closure = data.compute_boundary_closure()
process_external_elemental_closure_df = data.build_closure_summary_df(process_external_elemental_closure)

process_dry_biomass = next(
    spec for spec in process_external_elemental_closure['component_specs']
    if spec['label'] == 'DCM out'
 )

print('Whole-process external-boundary CHON closure:')
display(process_external_elemental_closure_df)
print(f"Max absolute elemental residual: {process_external_elemental_closure['max_abs_residual']:.6g} mol-atoms")

Whole-process external-boundary CHON closure:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,4.689631e+03,4.689588e+03,-0.043018,-0.000917
1,H,2.303179e+06,2.303180e+06,0.890067,0.000039
2,O,1.153179e+06,1.153288e+06,108.709276,0.009427
3,N,7.394463e+02,7.394395e+02,-0.006834,-0.000924


Max absolute elemental residual: 108.709 mol-atoms


## 4. Batch-State and Operating-Control Figures

Use `BATCH_NUMBER` from the configuration section to inspect biomass, metabolites, volume/feed, gas transfer, and pH control for one batch.

## 5. Batch Plots

In [33]:
# Cell density & metabolites
fig = data.plot_biomass_metabolites(batch_number=BATCH_NUMBER)
fig.show()

In [34]:
# Volume & feed / dilution rate
fig = data.plot_volume_feed(batch_number=BATCH_NUMBER)
fig.show()

In [35]:
# Gas transfer (DO, agitation, gas flow, O2 fraction)
fig = data.plot_gas_transfer(batch_number=BATCH_NUMBER)
fig.show()

In [36]:
# pH plots
fig = data.plot_pH_control(batch_number=BATCH_NUMBER)
fig.show()

## 5. Kinetic-Behavior Figures

Review nutrient limitation, inhibition, kinetic rates, and stoichiometric yields for the selected batch.

In [37]:
# Nutrient limitation factors (Monod terms)
fig = data.plot_nutrient_factors(batch_number=BATCH_NUMBER)
fig.show()

In [38]:
# Inhibition factors (lactate, ammonia, O2, osmolality)
fig = data.plot_inhibition_factors(batch_number=BATCH_NUMBER)
fig.show()

In [39]:
# Specific rates (mu, q_Glc, q_Gln, q_Lac, q_O2, etc.)
fig = data.plot_kinetic_rates(batch_number=BATCH_NUMBER)
fig.show()

In [40]:
# Stoichiometric yield coefficients (Y_X/Glc, Y_Lac/Glc, Y_O2/Glc, etc.)
fig = data.plot_stoichiometric_yields(batch_number=BATCH_NUMBER)
fig.show()

## 6. Multi-Batch Process Figures

Use `MAX_BATCHES` from the configuration section to constrain batch overlays and cumulative totals.

In [41]:
# Overlaid state variable profiles across batches
figs = data.plot_profiles(max_batches=MAX_BATCHES)
for fig in figs:
    fig.show()

In [42]:
# Cumulative harvest (biomass & volume) across batches
fig = data.plot_cumulative_harvest(max_batches=MAX_BATCHES)
fig.show()

In [43]:
# Cumulative medium use
fig = data.plot_cumulative_medium_use(max_batches=MAX_BATCHES)
fig.show()

In [44]:
# Cumulative gas use
fig = data.plot_cumulative_gas_use(max_batches=MAX_BATCHES)
fig.show()

## 7. Interpretation Checklist

Before accepting a result:

- Confirm the selected pickle and whether the optional re-run path was enabled.
- Investigate a failed optimization feasibility check, especially its rejection diagnostics and termination reason.
- Review elemental residuals in both closure reports; large residuals require mass-balance investigation.
- Check the selected-batch figures for unexpected depletion, volume limits, gas-control saturation, or pH-control behavior.
- If a plot lacks data, verify that the pickle was generated with `performance_mode=False` and contains detailed trajectories.